# Accuracy Test Runner

This cell below runs all cocotb accuracy tests via the `accuracy_tests/Makefile`.
Each test invokes `make -f Makefile results.xml` with the appropriate
`HLS_SOLN`, `TOPLEVEL`, and `MODULE` environment variables. Make sure to run this with the prog-synth-verif env. 

In [1]:
import os
import subprocess
from pathlib import Path

def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent:
        if (root / "accuracy_tests" / "Makefile").exists():
            return root
        root = root.parent
    raise RuntimeError("Could not locate repo root containing accuracy_tests/Makefile")

ROOT = find_repo_root()
ACC_DIR = ROOT / "accuracy_tests"

TESTS = [
    {
        "name": "mxint8_adder",
        "soln": "solution_mxint8addition_full_sum",
        "top": "add_full_sum",
        "module": "tests.addition.test_mxint8_adder",
    },
    {
        "name": "mxint8_multiplier",
        "soln": "solution_mxint8multiplication_full_product",
        "top": "mult_mxint_full_product",
        "module": "tests.multiplication.test_mxint8_multiplier",
    },
    {
        "name": "fp32_adder",
        "soln": "solution_fp32addition_fp32_full_sum",
        "top": "fp32_sum",
        "module": "tests.addition.test_fp32_adder",
    },
]

SHOW_FULL_LOG = False
TAIL_LINES = 200

def tail(text: str, lines: int) -> str:
    parts = text.splitlines()
    return "\n".join(parts[-lines:])

results = []

for t in TESTS:
    print(f"\n=== Running {t['name']} ===")
    env = os.environ.copy()
    env.update({
        "HLS_SOLN": t["soln"],
        "TOPLEVEL": t["top"],
        "MODULE": t["module"],
    })

    # Clean previous report if present.
    results_xml = ACC_DIR / "results.xml"
    if results_xml.exists():
        results_xml.unlink()

    proc = subprocess.run(
        ["make", "-f", "Makefile", "results.xml"],
        cwd=ACC_DIR,
        env=env,
        text=True,
        capture_output=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    if SHOW_FULL_LOG:
        print(stdout)
        if stderr:
            print(stderr)
    else:
        print(tail(stdout, TAIL_LINES))
        if stderr:
            print("\n[stderr tail]\n" + tail(stderr, TAIL_LINES))

    results.append({
        "name": t["name"],
        "returncode": proc.returncode,
    })

print("\n=== Summary ===")
for r in results:
    status = "PASS" if r["returncode"] == 0 else "FAIL"
    print(f"{r['name']}: {status}")



=== Running mxint8_adder ===
/usr/bin/iverilog -o /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/accuracy_tests/sim_build/add_full_sum_solution_mxint8addition_full_sum/sim.vvp -s add_full_sum -g2012 -f /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/accuracy_tests/sim_build/add_full_sum_solution_mxint8addition_full_sum/cmds.f  /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/results/HLS/solution_mxint8addition_full_sum/verilog_out/add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_23_1.v /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/results/HLS/solution_mxint8addition_full_sum/verilog_out/add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_24_1.v /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/results/HLS/solution_mxint8addition_full_sum/verilog_out/add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_95_1.v /home/joe/Desktop/Uni/Year_4/Dissertation/Program-

# Hardware Results

This cell runs all top-level hardware components for all data formats. Make sure to run this with the prog-synth env. 

In [3]:
import json
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent:
        if (root / 'results' / 'cpp').exists():
            return root
        root = root.parent
    raise RuntimeError('Could not locate repo root containing results/cpp')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.run_vitis_hls import run_vitis_hls

# Toggle implementation (Vivado) stage; True gives LUT/FF/Fmax.
RUN_IMPL = True

CPP_DIR = ROOT / 'results' / 'cpp'

wanted = [
    'solution_mxint8addition_full_sum.cpp',
    'solution_fp32addition_fp32_full_sum.cpp',
    'solution_mxint8multiplication_full_product.cpp',
    'solution_naivefp32adder_fp32_adder.cpp',
    'solution_naiveadder_int8_int_add.cpp',
    'solution_naiveadder_int32_int_add.cpp',
    'solution_naivefp32multiplier_fp32_mul.cpp',
    'solution_naivemultiplier_int32_int_mul.cpp',
    # Legacy filenames (if still present from earlier runs)
    'solution_naiveadder_int_add.cpp',
    'solution_naivefp32adder_full_adder.cpp',
    'solution_naivemultiplier_int_mul.cpp',
]

cpp_files = [CPP_DIR / name for name in wanted]
missing = [p.name for p in cpp_files if not p.exists()]
cpp_files = [p for p in cpp_files if p.exists()]

if missing:
    print('Skipping missing components:', ', '.join(missing))
if not cpp_files:
    raise RuntimeError(f'No requested solution files found under {CPP_DIR}')

print(f'Found {len(cpp_files)} components to run.')

# Run HLS/Vivado for each component.
for cpp in cpp_files:
    print(f'\n=== Hardware run: {cpp.name} ===')
    run_vitis_hls(str(cpp), impl=RUN_IMPL)


Skipping missing components: solution_naivefp32adder_fp32_adder.cpp, solution_naivefp32multiplier_fp32_mul.cpp, solution_naivemultiplier_int32_int_mul.cpp
Found 8 components to run.

=== Hardware run: solution_mxint8addition_full_sum.cpp ===
[INFO] Using top from filename hint: add_full_sum
[INFO] Generated HLS TCL script: /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/results/HLS/solution_mxint8addition_full_sum/hls.tcl
[INFO] Running Vitis HLS synthesis...

[SUCCESS] Vitis HLS run completed successfully.
[INFO] Staged 3 Verilog files into: /home/joe/Desktop/Uni/Year_4/Dissertation/Program-Synthesis-for-Efficient-ML/results/HLS/solution_mxint8addition_full_sum/verilog_out
[INFO] Candidate RTL modules: ['add_full_sum', 'add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_23_1', 'add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_24_1', 'add_full_sum_add_full_sum_Pipeline_VITIS_LOOP_95_1', 'add_full_sum_bitselect_1ns_24ns_5ns_1_1_1', 'add_full_sum_flow_control_loop_pi